In [1]:
import numpy as np
import pandas as pd

In [3]:
data = pd.read_csv(r'C:\Users\Mayur23\OneDrive\Desktop\pareet\Student Feedback Analysis System\data\feedback_cleaned.csv')
data['Open_feedback'] = data['Open_feedback'].fillna('')

In [5]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('vader_lexicon')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mayur23\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Mayur23\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Mayur23\AppData\Roaming\nltk_data...


In [10]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words and word.isalpha()]
    return tokens

data['Cleaned_tokens'] = data['Open_feedback'].apply(clean_text)
data[['Open_feedback', 'Cleaned_tokens']].head()

,Open_feedback,Cleaned_tokens
0,Session quality varied depending on the trainer.,"[session, quality, varied, depending, trainer]"
1,Would like the syllabus to be updated more fre...,"[would, like, syllabus, updated, frequently]"
2,"Loved the hands-on projects, they really built...","[loved, handson, projects, really, built, conf..."
3,"Average experience, some sessions were better ...","[average, experience, sessions, better, others]"
4,Faced frequent audio/video issues during classes.,"[faced, frequent, audiovideo, issues, classes]"


In [11]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if text.strip() == '':
        return 'No Feedback'
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

data['Sentiment'] = data['Open_feedback'].apply(get_sentiment)
data[['Open_feedback', 'Sentiment']].head(10)

,Open_feedback,Sentiment
0,Session quality varied depending on the trainer.,Neutral
1,Would like the syllabus to be updated more fre...,Positive
2,"Loved the hands-on projects, they really built...",Positive
3,"Average experience, some sessions were better ...",Positive
4,Faced frequent audio/video issues during classes.,Neutral
5,Well-paced sessions and very approachable trai...,Neutral
6,The portal for assignments was confusing and s...,Negative
7,The portal for assignments was confusing and s...,Negative
8,"Very satisfied, the mentors were patient and e...",Positive
9,Session quality varied depending on the trainer.,Neutral


In [13]:
data['Sentiment'].value_counts()

Sentiment
Positive       1083
Neutral         665
Negative        201
No Feedback      51
Name: count, dtype: int64

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

feedback_text = data[data['Open_feedback'].str.strip() != '']['Open_feedback']

tfidf = TfidfVectorizer(stop_words='english', max_features=20)
tfidf_matrix = tfidf.fit_transform(feedback_text)

keywords = tfidf.get_feature_names_out()
print(keywords)

['better' 'bit' 'concepts' 'content' 'decent' 'expected' 'experience'
 'felt' 'industry' 'modules' 'okay' 'overall' 'program' 'really' 'rushed'
 'session' 'sessions' 'topics' 'trainer' 'use']


In [16]:
positive_text = data[data['Sentiment'] == 'Positive']['Open_feedback']
negative_text = data[data['Sentiment'] == 'Negative']['Open_feedback']

tfidf_pos = TfidfVectorizer(stop_words='english', max_features=15)
tfidf_pos.fit(positive_text)
print("Top Positive Keywords:", tfidf_pos.get_feature_names_out())

tfidf_neg = TfidfVectorizer(stop_words='english', max_features=15)
tfidf_neg.fit(negative_text)
print("Top Negative Keywords:", tfidf_neg.get_feature_names_out())

Top Positive Keywords: ['better' 'concepts' 'content' 'expected' 'experience' 'felt' 'hands'
 'okay' 'overall' 'planned' 'program' 'rushed' 'sessions' 'topics'
 'trainer']
Top Negative Keywords: ['fast' 'hard' 'introduced' 'kept' 'lagging' 'live' 'moved' 'new' 'online'
 'pace' 'platform' 'quickly' 'sessions' 'struggled' 'topics']


In [17]:
data[data['Open_feedback'].str.contains('rushed', case=False, na=False)][['Open_feedback', 'Sentiment']]

,Open_feedback,Sentiment
10,"Wish there was more time per topic, felt rushe...",Positive
20,"Wish there was more time per topic, felt rushe...",Positive
24,"Content was okay, though a few topics felt rus...",Positive
26,"Content was okay, though a few topics felt rus...",Positive
30,"Wish there was more time per topic, felt rushe...",Positive
...,...,...
1910,"Wish there was more time per topic, felt rushe...",Positive
1926,"Content was okay, though a few topics felt rus...",Positive
1936,"Content was okay, though a few topics felt rus...",Positive
1946,"Wish there was more time per topic, felt rushe...",Positive


In [18]:
print(f"Comments containing 'rushed': {len(data[data['Open_feedback'].str.contains('rushed', case=False, na=False)])}")
print(f"All labeled Positive by VADER, despite describing a pacing complaint")

Comments containing 'rushed': 122
All labeled Positive by VADER, despite describing a pacing complaint
